# 01 · Load the mock source data into Turso
Creates the three source tables in Turso and loads one CSV batch (upsert, so batches can be re-applied safely):

* **batch 1**: initial load (60 customers, 20 products, 304 sales lines, with deliberate bad data)
* **batch 2**: changes, new rows, and sales for customers that do not exist yet
* **batch 3**: the missing customers arrive

Set `TURSO_DATABASE_URL` and `TURSO_AUTH_TOKEN` as Codespaces secrets to write to Turso. Without them this loads a local SQLite stand-in (`lake/source_local.db`).

In [1]:
import os
os.chdir("/workspaces/pythonProject/medallion_v2/notebooks")

In [2]:
import os
BATCH = int(os.getenv("BATCH", 1))              # 1, 2 or 3: edit here, then run the pipeline notebook after each batch
RESET = os.getenv("RESET_SOURCE") == "1"        # True drops the source tables first (start over)

In [3]:
%run -i ./00_config.ipynb

cwd : /workspaces/pythonProject/medallion_v2/notebooks
ROOT: /workspaces/pythonProject/medallion_v2
LAKE: /workspaces/pythonProject/medallion_v2/lake
rules folder exists: True | file there: True
rule library: 22 checks for 8 entities


In [4]:
def find(name):
    """Locate a project file whether the repo is laid out in folders or everything sits in one folder."""
    for d in (Path.cwd(), ROOT, ROOT / "data", ROOT / "sql"):
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"{name} not found next to the notebook, in data/ or in sql/")

conn = source_conn()
print("target:", "Turso" if TURSO_URL else "local SQLite stand-in")

if RESET:
    for t in ("sales", "products", "customers"): conn.execute(f"DROP TABLE IF EXISTS {t}")

ddl = "\n".join(l for l in find("schema.sql").read_text().splitlines() if not l.strip().startswith("--"))
for stmt in [s.strip() for s in ddl.split(";") if s.strip()]:
    conn.execute(stmt)
conn.commit()

target: Turso


In [5]:
KEYS = {"customers": "customer_id", "products": "product_id", "sales": "order_line_id"}

def load_csv(table, path):
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    cols = list(df.columns)
    rows = [tuple(None if v == "" else v for v in r) for r in df.itertuples(index=False)]   # blank -> NULL
    upd = ", ".join(f"{c} = excluded.{c}" for c in cols if c != KEYS[table])
    sql = (f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))}) "
           f"ON CONFLICT({KEYS[table]}) DO UPDATE SET {upd}")
    for r in rows: conn.execute(sql, r)
    conn.commit()
    return len(rows)

for table in ("customers", "products", "sales"):
    try:
        f = find(f"{table}_batch{BATCH}.csv")
    except FileNotFoundError:
        continue                                  # not every batch has a file for every table
    print(f"{table:10s} loaded {load_csv(table, f):4d} rows from {f.name}")
try: conn.sync()                                 # push/pull when using an embedded replica
except Exception: pass

customers  loaded   60 rows from customers_batch1.csv
products   loaded   20 rows from products_batch1.csv
sales      loaded  304 rows from sales_batch1.csv


In [6]:
for t in ("customers", "products", "sales"):
    cur = conn.cursor(); cur.execute(f"SELECT count(*), min(updated_at), max(updated_at) FROM {t}")
    print(t, cur.fetchone())

customers (60, '2026-09-02 08:45:40', '2026-10-30 14:13:55')
products (20, '2026-09-05 19:17:22', '2026-10-31 18:47:42')
sales (304, '2026-09-01 09:56:16', '2026-10-31 20:05:30')


### Alternative: import with the Turso CLI
Skip the loader and use the CLI instead (needs the `sqlite3` CLI; Turso requires the file to be in WAL mode). Blank CSV cells import as empty strings, which the Silver cleaning step treats as NULL.

```bash
sqlite3 source.db < sql/schema.sql
sqlite3 source.db "PRAGMA journal_mode=WAL;"
for t in customers products sales; do sqlite3 source.db ".mode csv" ".import --skip 1 data/${t}_batch1.csv $t"; done
turso db import source.db        # database is named after the file
```
Batches 2 and 3 are upserts, so apply those with this notebook.